# Arabic Text Paraphrasing with AraT5: 128-Token Experiment

This notebook contains one of the two CIS733 experiments. Selected outputs show the results recorded during the course run. Rerunning a cell replaces its recorded output with results from the current environment.


## Setup

Install the dependencies from `requirements.txt` and download the CSV as described in `data/README.md`. The default path is `data/paraphrase_data.csv`.


In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import torch
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoTokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
current_directory = Path.cwd().resolve()
REPO_ROOT = current_directory.parent if current_directory.name == "notebooks" else current_directory

configured_data_path = os.environ.get(
    "ARABIC_PARAPHRASE_DATA",
    str(REPO_ROOT / "data" / "paraphrase_data.csv"),
)
DATA_PATH = Path(configured_data_path).expanduser()
if not DATA_PATH.is_absolute():
    DATA_PATH = (REPO_ROOT / DATA_PATH).resolve()

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Follow data/README.md or set "
        "ARABIC_PARAPHRASE_DATA to the CSV location."
    )

df = pd.read_csv(DATA_PATH)
required_columns = {"source", "destination"}
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(
        f"Dataset is missing required columns: {', '.join(missing_columns)}"
    )

print(f"Loaded {len(df):,} sentence pairs from {DATA_PATH}")


In [ ]:
source_sentences = df["source"].astype(str)
target_sentences = df["destination"].astype(str)

# This reproduces the course experiments: 20% test, then 10% of the
# remaining 80% for evaluation. The effective split is 72%/8%/20%.
train_sentences, test_sentences, train_targets, test_targets = train_test_split(
    source_sentences,
    target_sentences,
    test_size=0.2,
    random_state=SEED,
)
train_sentences, eval_sentences, train_targets, eval_targets = train_test_split(
    train_sentences,
    train_targets,
    test_size=0.1,
    random_state=SEED,
)

print(
    f"train={len(train_sentences):,}, evaluation={len(eval_sentences):,}, "
    f"test={len(test_sentences):,}"
)


## Tokenization and datasets


In [ ]:
MAX_LENGTH = 128
MODEL_NAME = os.environ.get("ARAT5_MODEL_NAME", "UBC-NLP/AraT5-msa-small")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


def tokenize(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )


tokenized_train_inputs = tokenize(train_sentences)
tokenized_train_targets = tokenize(train_targets)
tokenized_eval_inputs = tokenize(eval_sentences)
tokenized_eval_targets = tokenize(eval_targets)
tokenized_test_inputs = tokenize(test_sentences)
tokenized_test_targets = tokenize(test_targets)


In [ ]:
class ParaphraseDataset(Dataset):
    def __init__(self, tokenized_inputs, tokenized_targets):
        self.input_ids = tokenized_inputs["input_ids"]
        self.attention_mask = tokenized_inputs["attention_mask"]
        self.labels = tokenized_targets["input_ids"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return {
            "input_ids": self.input_ids[index],
            "attention_mask": self.attention_mask[index],
            "labels": self.labels[index],
        }


train_dataset = ParaphraseDataset(tokenized_train_inputs, tokenized_train_targets)
eval_dataset = ParaphraseDataset(tokenized_eval_inputs, tokenized_eval_targets)
test_dataset = ParaphraseDataset(tokenized_test_inputs, tokenized_test_targets)


## AraT5 fine-tuning


In [ ]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
print(f"Base model: {MODEL_NAME}")


In [ ]:
MODEL_OUTPUT_DIR = REPO_ROOT / "models" / "arat5_128"
training_args = TrainingArguments(
    output_dir=str(MODEL_OUTPUT_DIR),
    num_train_epochs=30,
    auto_find_batch_size=True,
    logging_steps=15000,
    save_steps=15000,
    eval_strategy="epoch",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=lambda data: {
        "input_ids": torch.stack([item["input_ids"] for item in data]),
        "attention_mask": torch.stack([item["attention_mask"] for item in data]),
        "labels": torch.stack([item["labels"] for item in data]),
    },
)


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,No log,1.173911
2,No log,0.932047
3,No log,0.815683
4,1.764000,0.758341
5,1.764000,0.719855
6,1.764000,0.696381
7,0.841000,0.678992
8,0.841000,0.664719
9,0.841000,0.654182
10,0.743400,0.645978


TrainOutput(global_step=135000, training_loss=0.8016950593171296, metrics={'train_runtime': 36832.7435, 'train_samples_per_second': 29.322, 'train_steps_per_second': 3.665, 'total_flos': 4.698839973888e+16, 'train_loss': 0.8016950593171296, 'epoch': 30.0})

## Paraphrase generation

Run the training cell above, or set `ARABIC_PARAPHRASE_MODEL_DIR` to a compatible local checkpoint before running the cells below.


In [ ]:
checkpoint_override = os.environ.get("ARABIC_PARAPHRASE_MODEL_DIR")
if checkpoint_override:
    checkpoint_path = Path(checkpoint_override).expanduser().resolve()
    if not checkpoint_path.is_dir():
        raise FileNotFoundError(
            f"Model directory not found at {checkpoint_path}. Train the model first or "
            "set ARABIC_PARAPHRASE_MODEL_DIR to a valid checkpoint."
        )
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, use_fast=False)
    model = T5ForConditionalGeneration.from_pretrained(checkpoint_path)
    print(f"Loaded checkpoint from {checkpoint_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print(f"Using device: {device}")


In [ ]:
def paraphrase(example: str) -> str:
    tokenized_input = tokenizer(
        [example],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    input_ids = tokenized_input["input_ids"].to(device)
    attention_mask = tokenized_input["attention_mask"].to(device)

    with torch.no_grad():
        generated_outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=256,
            do_sample=True,
            top_k=120,
            top_p=0.95,
            early_stopping=True,
        )

    return " ".join(
        tokenizer.decode(output, skip_special_tokens=True)
        for output in generated_outputs
    )


### Recorded examples


In [ ]:
paraphrase('ومع ذلك، أرى أن العكس قد يكون صحيحا في الواقع.')

'ومع ذلك، أعتقد أنه يمكن أن تكون صحيحة.'

In [ ]:
paraphrase('وطوال سنوات عديدة، تحدثنا أيضا عن نظام متكامل للإدارة والتفتيش في مجال الزراعة')

'وفترات عديدة، تحدثنا أيضا عن نظام متكامل لإدارة والرقابة على الزراعة.'

In [ ]:
paraphrase('وهذا فرق آخر في إجراءات التصريف')

'هذه هي مناطق جديدة من تدابير التفريغ.'

In [ ]:
paraphrase('ويتمثل الهدف الرئيسي للتقرير في أن إزالة حسابات النفقات الممولة من قسم الضمان التابع للفريق العامل المعني بالشؤون الاقتصادية والاجتماعية هو عمل جيد')

'وتهدف التقرير الرئيسي لإزالة حسابات الإنفاق الممولة من البرنامج الإداري، الذي هو العمل الصحيح.'

In [ ]:
paraphrase('حيث تعتبر الكتابة من أهم الطرق المستخدمة في عملية التواصل بين الأفراد')

'من حيث الكتابة هو أحد الأدوات الرئيسية للتواصل بين الأفراد.'

In [ ]:
paraphrase('تقديم المكافآت المادية والمعنوية للأطفال المتفوقين، فذلك يساعد على تحفيز ورفع روح المنافسة بين الطلاب، كما يدفع الطالب لبذل جهد مضاعف للحصول على المكافئة')

'يقدمون مكافآت مادية والمعنوية لأطفالهم. وهذا يساعد على تشجيع وتعزيز الرغبة المنافسة بين الطلاب وحفزهم على بذل جهد مضاعف للحصول على المكافئة.'

In [ ]:
paraphrase('تعتبر المدرسة الابتدائية هي المكان الذي يبدأ فيه الطلاب ببناء هوياتهم الوطنية, وتكوين انتمائهم للمجتمع المحلي المصغر الذي ينتمون له، بالإضافة إلى المجتمع الدولي')


'تعتبر المدرسة الابتدائية مكانا من بناء هوية وطنية أو تأسيس تكوين الانتماء للمجتمع المحلي المصغر التي ينتمون لها، بالإضافة إلى المجتمع الدولي.'

### Additional long-sentence examples


In [ ]:
paraphrase('يختلف أسلوب تعليم الأطفال وطريقة تقديم المعلومات في الصفوف الابتدائيّ يختلف أسلوب تعليم الأطفال وطريقة تقديم المعلومات في الصفوف الابتدائيّة؛ ويرجع ذلك لقصور قدرات الأطفال العقليّة')

'الاختلاف في مناهج تعليم الأطفال وكيفية تقديم المعلومات في الصف الابتدائي المختلفة طريقة تعليم الأطفال وبكيفية تقديم المعلومات في صفوف الابتدائي: هناك إلى حد كبير قصور قدرات الأطفال العقلي.'

In [ ]:
paraphrase('أثبتت الدراسات الحديثة أن البكاء مفيد للصحة، إلا أن نوعية الدموع التي تذرف نتيجة لموقف عاطفي انفعالي تكون مختلفة من حيث التركيبة والوظيفة؛ فالدموع عبارة عن تفاعلات كيميائية تأتي استجابة لدواع وحالات انفعالية مشبوبة بالعاطفة وبالتالي فهي أغنى بالبروتينات')


'تم إثبات البحث الحديث عن أن الشعور بصدمة صحية صحية، ولكن نوعية الدموع التي تتلقى نتيجة موقفها العاطفي عن طريق الردود العقلية المختلفة، وهو شعور من تفاعلات كيميائية التي يأتي استجابة لضوضات الطبيعة، وحالات نفسية متوازنة وتتكامل مع البروتينات.'

## BLEU evaluation

The first calculation retains the original course procedure for comparison with the recorded result. The second calculation uses the destination paraphrases as references and is the recommended evaluation for a new run.


In [ ]:
# Generating the full test set is expensive and requires a trained checkpoint.
test_sources = test_sentences.tolist()
test_references = test_targets.tolist()
generated_paraphrases = [paraphrase(sentence) for sentence in test_sources]
print(f"Generated {len(generated_paraphrases):,} paraphrases")


In [ ]:
def calculate_bleu_score(references, hypotheses):
    smoothing = SmoothingFunction().method1
    tokenized_references = [[reference.split()] for reference in references]
    tokenized_hypotheses = [hypothesis.split() for hypothesis in hypotheses]
    return corpus_bleu(
        tokenized_references,
        tokenized_hypotheses,
        smoothing_function=smoothing,
    )


# This reproduces the BLEU calculation used for the recorded course result.
# It compares generated text with the source sentences.
bleu_score = calculate_bleu_score(test_sources, generated_paraphrases)
print("BLEU Score:", bleu_score)


BLEU Score: 0.21093420727002457


In [ ]:
# For a standard paraphrase evaluation, compare generated text with the
# destination paraphrases from the test split.
reference_bleu_score = calculate_bleu_score(test_references, generated_paraphrases)
print("Reference BLEU Score:", reference_bleu_score)
